In [5]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(),'..'))
import CIBUSmod as cm

# Soil carbon and climate impact calculations
The code in this notebook calculates and manages C flows to and from soils as well as climate impacts.

In this section the `xarray` package is being heavily used for its excellent handling of multidimensional data.
It may take some time to get used to how to use it. For that reason, below is a cell with tips on how to learn using its functionalities.

For the functions and methods in the `SoilData` class to work the `xarray` package must be installed (pip install xarray)

## Run either Option 1 or Option 2 below

### Option 1: Create a new instance from a saved scenario

In [40]:
# To create a new 'SoilData' instance based on a session saved in 'run_scn' use:
FAI_new = cm.SoilData('session_FAC.csv', 'session_FAC_name')

Dataset set to ['input', 'soc', 'historic', 'total_soc']
input_df saved as FAC_input_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
Saved instance variable states to C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results/FAC


### Option 2: Recreate a saved instance that was computed and saved previously

In [23]:
# To restore and continue working on a previously created instance of the 'SoilData' class set the name and parameters of the following function call to the scenario name (e.g. 'FAI'):
FAI_new = cm.SoilData.load_instance_state('FAI_no_cows')
FAI_new.load_inventory()
#FAI_soil.save_inventory()
#FAI_soil.save_instance_state()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ÄGARE\\Documents\\PycharmProjects\\CIBUSmod\\data\\soil\\temp_results\\FAI_no_cows_input_ds.nc'

The following code block can be used to check which variables have currently been set.
Both public and private attributes are shown to facilitate quick troubleshooting. 

In [25]:
# 'private' functions are helper functions used to calculate the inventories and dataframes
# stored in the 'public'' variables. These are not saved and reloaded with the above commands
FAI_new.check_attributes_status('public') #, FAI_new.check_attributes_status('private')

('The following attributes are set for public variables',
 {'historic_ha_df': ('NoneType', False),
  'historic_inventory': ('NoneType', False),
  'historic_sko_df': ('NoneType', False),
  'input_df': ('DataFrame', True),
  'input_inventory': ('NoneType', False),
  'name': ('str', True),
  'scenario': ('str', True),
  'set_df_and_name': ('function', True),
  'soc_ha_df': ('NoneType', False),
  'soc_inventory': ('NoneType', False),
  'soc_sko_df': ('NoneType', False),
  'ss_input_df': ('NoneType', False),
  'startyear': ('int32', True),
  'total_soc_inventory': ('NoneType', False)})

**To check if the imported variables were set as expected you can run:**

In [41]:
scn_name = FAI_new
print(f'Name: {scn_name.name}')
print(f'Scenario: {scn_name.scenario}')
print(f'startyear: {scn_name.startyear}')
print(f'Index:\n{scn_name.input_df.index.dtypes}')
print(f'Columns:\n{scn_name.input_df.columns}')

Name: food_as_culture
Scenario: FAC
startyear: 2020
Index:
scn                    object
crop                   object
prod_system            object
region                  int64
input_year     datetime64[ns]
dtype: object
Columns:
Index(['area_ha', 'harvest_kgdm', 'crop_residues_harvest_kgdm',
       'manure_cattle_kgC', 'manure_horses_kgC', 'manure_pigs_kgC',
       'manure_poultry_kgC', 'manure_sheep_kgC'],
      dtype='object')


# Run the basic calculations in three steps: input/soc/historic

## Calculate the carbon inputs for each input fraction in the input_df
The `calc_scn_inputs` method calculates the C input for each fraction of `input_df` and sets the  following variable attributes:
- `startyear`: defined as the first year of the timeseries in `input_df`.
- `input_inventory`: xarray dataset with `scn, crop, prod_system, region, input_year` as coordinates, including all original input and the C input per ha as well as per SKO for all fractions.
- `ss_input_df.columns`: dataframe containing the SOC inputs to use for the spinup modelling, corresponding to the C inputs for all fraction in the `startyear`, also both per ha and per SKO. 

**Note:** the column or index level name `year` in the `input_df` will be renamed to `input_year` to distinguish it from other time coordinates calculated with ICBM and subsequent temperature response functions.

In [42]:
FAI_new.calc_scn_inputs(verbose=False, looped=True)

Calculating scenario inputs...
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)
Warning, first index label is None, conversion of index label not possible
Warning, index label is None. No conversion of index values (should be auto-generated ints)


C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\notebooks\..\CIBUSmod\soil_modules\soil_utils.py:543: RuntimeWarning: invalid value encountered in scalar multiply
  i_ag_1 = (param_df.iloc[:,0][crop] + param_df.iloc[:,1][crop] * H)
C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\notebooks\..\CIBUSmod\soil_modules\soil_utils.py:544: RuntimeWarning: invalid value encountered in scalar multiply
  i_ag_2 = + (param_df.iloc[:,2][crop] + param_df.iloc[:,3][crop] * H)
C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\notebooks\..\CIBUSmod\soil_modules\soil_utils.py:546: RuntimeWarning: invalid value encountered in scalar multiply
  i_bg = + (param_df.iloc[:,4][crop] + param_df.iloc[:,5][crop] * H)


In [43]:
FAI_new.input_df.index.dtypes, FAI_new.input_df.dtypes

(scn                    object
 crop                   object
 prod_system            object
 region                  int64
 input_year     datetime64[ns]
 dtype: object,
 area_ha                    float64
 harvest_kgdm               float64
 crop_residues_kgdm         float64
 i_ag_manure_cattle_kgc     float64
 i_ag_manure_horses_kgc     float64
 i_ag_manure_pigs_kgc       float64
 i_ag_manure_poultry_kgc    float64
 i_ag_manure_sheep_kgc      float64
 areayield                  float64
 areayield_residues         float64
 i_ag_manure_cattle_ha      float64
 i_ag_manure_horses_ha      float64
 i_ag_manure_pigs_ha        float64
 i_ag_manure_poultry_ha     float64
 i_ag_manure_sheep_ha       float64
 i_ag_crop_ha               float64
 i_bg_crop_ha               float64
 alloc_source_crop           object
 i_ag_crop_kgc              float64
 i_bg_crop_kgc              float64
 dtype: object)

In [44]:
# To check that the dataseries has been calculated use
FAI_new.input_inventory.sel({'crop': 'rye'})

<xarray.Dataset> Size: 1MB
Dimensions:                  (scn: 1, prod_system: 2, region: 106,
                              input_year: 31)
Coordinates:
  * scn                      (scn) object 8B 'fac'
    crop                     <U3 12B 'rye'
  * prod_system              (prod_system) object 16B 'conventional' 'organic'
  * region                   (region) int64 848B 111 112 311 ... 2512 2519 2521
  * input_year               (input_year) datetime64[ns] 248B 2020-01-01 ... ...
Data variables: (12/20)
    area_ha                  (scn, prod_system, region, input_year) float64 53kB ...
    harvest_kgdm             (scn, prod_system, region, input_year) float64 53kB ...
    crop_residues_kgdm       (scn, prod_system, region, input_year) float64 53kB ...
    i_ag_manure_cattle_kgc   (scn, prod_system, region, input_year) float64 53kB ...
    i_ag_manure_horses_kgc   (scn, prod_system, region, input_year) float64 53kB ...
    i_ag_manure_pigs_kgc     (scn, prod_system, region, input_year) float64 53kB ...
    ...                       ...
    i_ag_manure_sheep_ha     (scn, prod_system, region, input_year) float64 53kB ...
    i_ag_crop_ha             (scn, prod_system, region, input_year) float64 53kB ...
    i_bg_crop_ha             (scn, prod_system, region, input_year) float64 53kB ...
    alloc_source_crop        (scn, prod_system, region, input_year) object 53kB ...
    i_ag_crop_kgc            (scn, prod_system, region, input_year) float64 53kB ...
    i_bg_crop_kgc            (scn, prod_system, region, input_year) float64 53kB ...

In [45]:
# Save the data that was just calculated to enable the use of option 2, above
FAI_new.save_inventory('input')

input_inventory saved as FAC_input_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
input_df saved as FAC_input_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
ss_input_df saved as FAC_ss_input_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results


## Calculate the SOC timeseries
The `calc_soc_timeseries` currently calculates the soc timeseries for each individual yearly input in the `input_inventory`. This is a memory intensive operation. It may cause the jupyter instance to crash if memory allocation is not sufficiently large.

**TODO:** Redefine function to operate on fractions of the database to reduce memory load.

In [46]:
# Add 'crop' to the group to calculate per crop, otherwise all inputs are summed for all crops in a region
# NOTE: adding crop increases computation time proportionately to the number of crops.
# Original computation time (1m 1s 427ms, 55s 585ms)
FAI_new.calc_soc_timeseries(group=['scn', 'prod_system', 'region', 'input_year'], verbose=False, looped=True)

Calculating SOC timeseries...


In [47]:
# To check that the dataseries has been calculated use
FAI_new.soc_inventory

<xarray.Dataset> Size: 221MB
Dimensions:      (scn: 1, prod_system: 2, region: 106, input_year: 31,
                  output_year: 100, fraction: 14)
Coordinates:
  * scn          (scn) object 8B 'fac'
  * prod_system  (prod_system) object 16B 'conventional' 'organic'
  * region       (region) int64 848B 111 112 311 312 321 ... 2511 2512 2519 2521
  * input_year   (input_year) datetime64[ns] 248B 2020-01-01 ... 2050-01-01
  * output_year  (output_year) datetime64[ns] 800B 2020-01-01 ... 2119-01-01
  * fraction     (fraction) object 112B 'i_ag_crop_ha' ... 'i_bg_crop_kgc'
Data variables:
    input_area   (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    y_pool       (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...
    o_pool       (scn, prod_system, region, input_year, output_year, fraction) float64 74MB ...

In [48]:
# Save the data just calculated to enable the use of option 2, above
FAI_new.save_inventory('soc')

soc_inventory saved as FAC_soc_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
soc_ha_df saved as FAC_soc_ha_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
soc_sko_df saved as FAC_soc_sko_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results


## Calculate the historic SOC timeseries
The `calc_historic_soc_timeseries` calculates the soc timeseries for each individual SS input in the `ss_input_df`. The inputs all take place in the year 2020 (and only this year), making the computation much lighter and faster than the previous ones.


In [49]:
# original code execution time (2s 842ms)
FAI_new.calc_historic_soc_timeseries(verbose=False)

Calculating historic SOC timeseries...
---Leaving _calculate_historic_soc()---


In [50]:
# To check that the dataseries has been calculated use
FAI_new.historic_inventory

<xarray.Dataset> Size: 7MB
Dimensions:      (prod_system: 2, region: 106, fraction: 14, output_year: 100)
Coordinates:
  * prod_system  (prod_system) object 16B 'conventional' 'organic'
  * region       (region) int64 848B 111 112 311 312 321 ... 2511 2512 2519 2521
  * fraction     (fraction) object 112B 'i_ag_crop_ha' ... 'i_bg_crop_kgc'
  * output_year  (output_year) datetime64[ns] 800B 2020-01-01 ... 2119-01-01
Data variables:
    input_area   (prod_system, region, fraction, output_year) float64 2MB 2.7...
    y_pool       (prod_system, region, fraction, output_year) float64 2MB 3.0...
    o_pool       (prod_system, region, fraction, output_year) float64 2MB 1.3...

In [51]:
# Save the data just calculated to enable the use of option 2, above
FAI_new.save_inventory('historic')

historic_inventory saved as FAC_historic_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
historic_ha_df saved as FAC_historic_ha_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
historic_sko_df saved as FAC_historic_sko_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results


## Calculate and assign the tot SOC and CO2 fluxes, then combine the datasets
When both future and historic SOC timeseries have been calculated total SOC and annual CO2 fluxes remain to be calculated.
These will be added as new data arrays to the existing `soc_inventory` and `historic_inventory` datasets which are then combined into a new dataset: `total_soc_inventory` when running the following method

In [52]:
FAI_new.total_merge()

In [53]:
# Save the data just calculated to enable the use of option 2, above
FAI_new.save_inventory('total_soc')

total_soc_inventory saved as FAC_total_soc_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results


## Saving and loading datasets
The `SoilData` class has instance methods to save and read saved datasets to avoid having to rerun the computations between sessions. These are **NOT** automatic, but have to be invoked by the user.

The `save_inventory` and `load_inventory` methods can be called with three optional strings:

- `inputs` saves and loads the input inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The  netcdf file is named `<scenario_name>_input_ds.nc`
- `soc` saves and loads the soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_soc_ds.nc`
- `historic` saves and loads the historic soc inventory dataset to/from `CIBUSmod/data/soil/temp_results`. The netcdf file is named `<scenario_name>_historic_soc_ds.nc`

The `save_instance_state` saves all non dataframe/dataset variables in a pickle file. This needs to be run to include all other set variable states. Without them many methods will not work.  



In [54]:
# The commands in this cell saves the current state of the SoilData instance
FAI_new.save_inventory()
FAI_new.save_instance_state()

Dataset set to ['input', 'soc', 'historic', 'total_soc']
input_inventory saved as FAC_input_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
input_df saved as FAC_input_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
ss_input_df saved as FAC_ss_input_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
soc_inventory saved as FAC_soc_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
soc_ha_df saved as FAC_soc_ha_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
soc_sko_df saved as FAC_soc_sko_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
historic_inventory saved as FAC_historic_ds.nc in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
historic_ha_df saved as FAC_historic_ha_df in C:\Users\ÄGARE\Documents\PycharmProjects\CIBUSmod\data\soil\temp_results
historic_sko_df saved as FAC_hi


 # After having run this notebook and saved inventory and state the data will be available in other notebooks by running the statements below
#### These commands instantiates and sets all the variable states of the instance to what it was when it was previously saved
FAI = cm.SoilData.load_instance_state('FAI')
FAI.load_inventory()

#### For analysing the data, use the SoilDataExplore class. This can be instatiated like this:
FAI = cm.SoilDataExplore('FAI')

### given that CIBUSmod has been imported as cm